# check_paralle.ipynb

Run this **after the first VM has started `training.py`** to confirm the switch to the
coordinated multi-VM mode worked. It checks four things:

1. **O_EXCL works on this MooseFS mount** — the whole claim protocol rests on an exclusive
   create being atomic. (Single-VM proves the mechanism; true cross-client needs 2 VMs.)
2. **Coordination is active** — this VM registered in `VM_parallel/` with a *fresh lease*.
3. **Migration succeeded** — combos the pre-migration run FINISHED (manifest `status=done`)
   are recognized as done and not re-run.
4. **Live progress** — done / in-flight / resuming / free, and by which VM.

**"done" = the trainer's manifest says `status=done` (all epochs finished), NOT just that a
checkpoint file exists** — `vicreg_review_h5_latest.pt` is the *resumable* checkpoint written
every epoch, so an in-flight or interrupted combo has one too (and correctly gets resumed).

In [ ]:
# Config -- point at the SHARED coordinated out_dir (the same dir the current run's
# checkpoints already live in). Must match training.ipynb's OUT_DIR.
import sys, os, json, time
from pathlib import Path

REPO = '/workspace/stable-query-latent'
OUT_DIR = os.path.join(REPO, 'VICReg_review/heads/cloud_full_sweep_a100')
SWEEP_YAML = os.path.join(REPO, 'VICReg_review/sweep/sweep.yaml')
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from VICReg_review.sweep.config import SweepConfig
from VICReg_review.sweep import jobspec
from VICReg_review.sweep import coordination as coord

ROOT = Path(OUT_DIR)
print('repo    :', REPO)
print('out_dir :', OUT_DIR, '(exists:', ROOT.exists(), ')')

In [ ]:
# 1. O_EXCL atomicity self-test on THIS filesystem (the claim primitive).
#    Create a file exclusively twice -> the second MUST fail. If it doesn't, the
#    network FS is not honouring exclusive create and coordination is unsafe here.
probe = ROOT / '_coord_probe'
probe.mkdir(parents=True, exist_ok=True)
tf = probe / f'oexcl_{int(time.time()*1000)}.tmp'
first = coord._atomic_create(tf, {'ok': 1})
second = coord._atomic_create(tf, {'ok': 2})
try:
    tf.unlink()
except OSError:
    pass
print('O_EXCL create: first =', first, ' second =', second)
print('VERDICT:', 'OK (exclusive create is atomic here)' if (first and not second)
      else 'FAIL -- exclusive create is NOT reliable on this mount; coordination unsafe')

In [ ]:
# 2. Coordination active? List registered VMs + whether their lease is fresh.
vm_dir = ROOT / coord.VM_DIR
vms = sorted(vm_dir.glob('*.json')) if vm_dir.exists() else []
if not vms:
    print('NO VM_parallel/ entries found ->',
          'coordination is NOT active. Either training.py is not the coordinated build,',
          'or no VM has started yet.')
else:
    now = time.time()
    print(f'{len(vms)} VM(s) registered in {vm_dir}:')
    for f in vms:
        rec = coord._read_json(f) or {}
        exp = rec.get('expiry')
        if exp is not None:
            fresh = f'lease {exp - now:+.0f}s' + (' (FRESH)' if exp > now else ' (EXPIRED)')
        else:
            age = now - f.stat().st_mtime
            fresh = f'mtime {age:.0f}s ago'
        print(f"  - {rec.get('vm', f.stem):20} pid={rec.get('pid')} {fresh}  info={rec.get('info', {})}")

In [ ]:
# 3 + 4. Migration + live progress. For every combo, classify by the manifest (the
#        authoritative finished-signal) + coordination markers:
#   done      = manifest status=done  OR  done.json         (finished; skipped)
#   in-flight = a live VM's status.json claim, not yet done  (may have a partial ckpt)
#   resuming  = a partial latest.pt, no claim, not done      (will be re-claimed + resumed)
#   free      = nothing yet
def manifest_status(cdir):
    try:
        return json.loads((cdir / 'vicreg_review_h5_manifest.json').read_text()).get('status')
    except Exception:
        return None

cfg = SweepConfig.load(SWEEP_YAML)
cfg.out_dir = OUT_DIR
combos = list(cfg.iter_combos())

done = inflight = resuming = free = 0
inflight_by = {}
migration_bad = []   # a FINISHED combo being re-claimed (real failure -> should be empty)
for c in combos:
    cdir = ROOT / c.combo_id
    is_done = manifest_status(cdir) == 'done' or (cdir / coord.DONE).exists()
    has_status = (cdir / coord.STATUS).exists()
    has_ckpt = (cdir / 'vicreg_review_h5_latest.pt').exists()
    if is_done:
        done += 1
        if has_status and not (cdir / coord.DONE).exists():
            migration_bad.append(c.combo_id)     # finished (manifest) yet claimed again
    elif has_status:
        inflight += 1
        st = coord._read_json(cdir / coord.STATUS) or {}
        inflight_by[st.get('vm', '?')] = inflight_by.get(st.get('vm', '?'), 0) + 1
    elif has_ckpt:
        resuming += 1
    else:
        free += 1

print(f'grid      : {len(combos)} combos')
print(f'done      : {done}   (manifest status=done or done.json)')
print(f'in-flight : {inflight}  by {inflight_by}   <- being trained now (partial ckpt is normal)')
print(f'resuming  : {resuming}   (partial checkpoint, not claimed -> will resume, NOT skipped)')
print(f'free      : {free}')
print()
if migration_bad:
    print('MIGRATION FAIL: these FINISHED combos are being re-claimed (real problem):')
    print('  ', migration_bad[:12], '...' if len(migration_bad) > 12 else '')
else:
    print('MIGRATION OK: no finished combo is being re-run; in-flight/resuming combos are expected.')